# SML Benchmark Evaluation

Runs [EleutherAI lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) on the fine-tuned SML model (`sweetpapa/sml-qwen2.5-3b-phase2`) **with** and **without** SML context injection.

**What this notebook does:**
1. Installs all dependencies (lm-eval, spaCy, etc.)
2. Clones the SML repo to get the Bible DB (50 MB SQLite)
3. Defines all SML modules inline (encoder, formatter, Bible query, harness wrapper)
4. Runs **baseline** eval (plain model, no SML) on `arc_challenge`, `hellaswag`, `truthfulqa_mc2`, `winogrande`
5. Runs **SML-augmented** eval (SML context injected into every prompt)
6. Prints a comparison table with deltas

**Runtime:** ~30-60 min on T4, ~15-30 min on A100. Use `LIMIT` to cap examples per task for faster iteration.

## 0. Environment Check

In [ ]:
import subprocess, sys, time

# Check GPU
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"GPU: {gpu_info[0] if gpu_info else 'NONE — enable GPU in Runtime > Change runtime type'}")
print(f"Python: {sys.version}")

# Check CUDA
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")

## 1. Install Dependencies

In [ ]:
%%time
!pip install -q lm-eval>=0.4.4 spacy>=3.7.0 tqdm
!python -m spacy download en_core_web_sm -q
print("Dependencies installed.")

## 2. Get Bible DB from Google Drive

In [ ]:
%%time
import os
from google.colab import drive

BIBLE_DB_PATH = "/content/sml_bible.db"

if not os.path.exists(BIBLE_DB_PATH):
    print("Mounting Google Drive...")
    drive.mount("/content/drive")
    src = "/content/drive/MyDrive/spt-ml-test/sml_bible.db"
    assert os.path.exists(src), f"Bible DB not found at {src}"
    print("Copying Bible DB to local disk (faster I/O)...")
    !cp "{src}" {BIBLE_DB_PATH}
    print(f"Bible DB ready: {BIBLE_DB_PATH}")
else:
    print(f"Bible DB already exists: {BIBLE_DB_PATH}")

print(f"Size: {os.path.getsize(BIBLE_DB_PATH) / 1e6:.1f} MB")

## 3. Define SML Modules Inline

All SML code is embedded here so nothing else needs to be cloned or installed.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SML CONFIG (constants only — no file paths, no .env)
# ═══════════════════════════════════════════════════════════════════════

RELATION_TYPES = {
    1: "IsA", 2: "PartOf", 3: "HasA", 4: "HasProperty", 5: "CapableOf",
    6: "AtLocation", 7: "Causes", 8: "HasPrerequisite", 9: "HasFirstSubevent",
    10: "HasLastSubevent", 11: "MotivatedByGoal", 12: "UsedFor", 13: "CreatedBy",
    14: "DefinedAs", 15: "SymbolOf", 16: "MadeOf", 17: "ReceivesAction",
    18: "Desires", 19: "CausesDesire", 20: "HasContext", 21: "SimilarTo",
    22: "Antonym", 23: "DerivedFrom", 24: "RelatedTo", 25: "FormOf",
    26: "EtymologicallyRelatedTo", 27: "Synonym", 28: "MannerOf",
    29: "LocatedNear", 30: "HasContext", 31: "dbpedia/genre",
    32: "dbpedia/occupation", 33: "dbpedia/language", 34: "dbpedia/capital",
}
RELATION_TYPES_INV = {v: k for k, v in RELATION_TYPES.items()}

print("Config loaded.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SML FORMATTER
# ═══════════════════════════════════════════════════════════════════════

def format_eda(array):
    return f"E({array[0]}|{array[1]}|{array[2]}|{array[3]}|{array[4]}|{array[5]}|{array[6]}|{array[7]})"

def format_ra(array):
    rel_type = array[0]
    negation = array[5]
    label = RELATION_TYPES.get(rel_type, str(rel_type)) if isinstance(rel_type, int) else str(rel_type)
    if negation and str(negation) not in ("0", "0.0", "False"):
        label = f"NOT_{label}"
    return f"R({label}|{array[1]}|{array[2]}|{array[3]}|{array[4]}|0)"

def format_sml_block(entities, relations):
    lines = [format_eda(e) for e in entities] + [format_ra(r) for r in relations]
    return f"<sml>\n" + "\n".join(lines) + "\n</sml>"

def parse_sml_block(sml_text):
    entities, relations = [], []
    content = sml_text.strip()
    if content.startswith("<sml>"): content = content[5:]
    if content.endswith("</sml>"): content = content[:-6]
    for line in content.strip().split("\n"):
        line = line.strip()
        if not line: continue
        if line.startswith("E(") and line.endswith(")"):
            parts = line[2:-1].split("|")
            parsed = []
            for p in parts:
                try: parsed.append(int(p))
                except ValueError:
                    try: parsed.append(float(p))
                    except ValueError: parsed.append(p)
            entities.append(parsed)
        elif line.startswith("R(") and line.endswith(")"):
            parts = line[2:-1].split("|")
            parsed, negation = [], 0
            for idx, p in enumerate(parts):
                if idx == 0:
                    rel_label = p
                    if p.startswith("NOT_"): negation = 1; rel_label = p[4:]
                    if rel_label in RELATION_TYPES_INV: parsed.append(RELATION_TYPES_INV[rel_label])
                    else:
                        try: parsed.append(int(rel_label))
                        except ValueError: parsed.append(rel_label)
                else:
                    try: parsed.append(int(p))
                    except ValueError:
                        try: parsed.append(float(p))
                        except ValueError: parsed.append(p)
            if len(parsed) == 6: parsed[5] = negation
            relations.append(parsed)
    return {"entities": entities, "relations": relations}

print("Formatter loaded.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BIBLE QUERY
# ═══════════════════════════════════════════════════════════════════════

import sqlite3

class Bible:
    def __init__(self, db_path):
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row

    def close(self):
        self.conn.close()

    def lookup_concept(self, text):
        row = self.conn.execute(
            "SELECT * FROM concepts WHERE LOWER(surface_text) = LOWER(?)", (text,)
        ).fetchone()
        return dict(row) if row else None

    def get_outgoing_relations(self, concept_id):
        rows = self.conn.execute(
            """SELECT r.*, rt.label as relation_label,
                      c.surface_text as target_text, c.anchor_token as target_anchor
               FROM relations r
               JOIN relation_types rt ON r.relation_type_id = rt.id
               JOIN concepts c ON r.target_id = c.id
               WHERE r.source_id = ?""",
            (concept_id,),
        ).fetchall()
        return [dict(r) for r in rows]

    def search_fuzzy(self, text, limit=5):
        sanitized = "".join(c for c in text if c.isalnum() or c in (" ", "_"))
        sanitized = sanitized.strip()
        if not sanitized: return []
        tokens = sanitized.split()
        if not tokens: return []
        match_expr = " ".join(f'"{ t}"' for t in tokens[:-1])
        match_expr = (match_expr + " " if match_expr else "") + f'"{tokens[-1]}"*'
        try:
            rows = self.conn.execute(
                """SELECT c.* FROM concepts_fts fts
                   JOIN concepts c ON fts.rowid = c.id
                   WHERE concepts_fts MATCH ?
                   ORDER BY rank LIMIT ?""",
                (match_expr, limit),
            ).fetchall()
        except Exception:
            return []
        return [dict(r) for r in rows]

    def get_concept_by_id(self, concept_id):
        row = self.conn.execute(
            "SELECT * FROM concepts WHERE id = ?", (concept_id,)
        ).fetchone()
        return dict(row) if row else None

# Quick sanity check
b = Bible(BIBLE_DB_PATH)
n = b.conn.execute("SELECT COUNT(*) FROM concepts").fetchone()[0]
print(f"Bible loaded: {n:,} concepts")
b.close()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SML ENCODER
# ═══════════════════════════════════════════════════════════════════════

import spacy

_STOP_WORDS = frozenset({
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "do", "does", "did", "have", "has", "had", "can", "could", "will",
    "would", "shall", "should", "may", "might", "must",
    "what", "which", "who", "whom", "where", "when", "why", "how",
    "you", "your", "i", "me", "my", "we", "our", "he", "she", "it",
    "they", "them", "their", "this", "that", "these", "those",
    "not", "no", "nor", "and", "or", "but", "if", "then", "so",
    "of", "in", "on", "at", "to", "for", "with", "by", "from",
    "up", "about", "into", "through", "during", "before", "after",
    "above", "below", "between", "out", "off", "over", "under",
    "again", "further", "too", "very", "just", "also", "now",
    "there", "here", "all", "each", "every", "both", "few", "more",
    "most", "other", "some", "such", "only", "own", "same",
})


class SMLEncoder:
    def __init__(self, bible_path, spacy_model="en_core_web_sm"):
        self.nlp = spacy.load(spacy_model)
        self.bible = Bible(bible_path)
        self._embedder = None

    def close(self):
        self.bible.close()

    def _resolve_concept(self, token_text, context=""):
        if token_text.lower() in _STOP_WORDS:
            return None
        concept = self.bible.lookup_concept(token_text.lower())
        if concept: return concept
        doc = self.nlp(token_text)
        if doc and doc[0].lemma_ != token_text.lower():
            concept = self.bible.lookup_concept(doc[0].lemma_)
            if concept: return concept
        results = self.bible.search_fuzzy(token_text.lower(), limit=3)
        return results[0] if results else None

    def _make_unknown_eda(self, word):
        return [0, 0, 0, 0, f"unknown_{word.lower().replace(' ', '_')}", 0, 0, 0.3]

    def _concept_to_eda(self, concept, modifiers=None, confidence=0.9):
        mod1 = modifiers[0]["anchor_token"] if modifiers and len(modifiers) > 0 else 0
        mod2 = modifiers[1]["anchor_token"] if modifiers and len(modifiers) > 1 else 0
        return [
            concept["domain"], concept["category"], concept["subcategory"],
            concept["specificity"], concept["anchor_token"],
            mod1, mod2, round(confidence, 2),
        ]

    def _get_bible_modifiers(self, concept, text, max_modifiers=2):
        rels = self.bible.get_outgoing_relations(concept["id"])
        has_prop = [r for r in rels if r["relation_type_id"] == 4]
        if not has_prop: return []
        has_prop.sort(key=lambda r: r["weight"], reverse=True)
        mods = []
        for rel in has_prop[:max_modifiers]:
            target = self.bible.get_concept_by_id(rel["target_id"])
            if target: mods.append(target)
        return mods

    def _find_bible_relations(self, entity_concepts):
        relations, seen = [], set()
        for i, ci in enumerate(entity_concepts):
            if ci is None: continue
            outgoing = self.bible.get_outgoing_relations(ci["id"])
            target_map = {}
            for rel in outgoing:
                tid = rel["target_id"]
                if tid not in target_map or rel["weight"] > target_map[tid]["weight"]:
                    target_map[tid] = rel
            for j, cj in enumerate(entity_concepts):
                if i == j or cj is None or (i, j) in seen: continue
                if cj["id"] in target_map:
                    rel = target_map[cj["id"]]
                    seen.add((i, j))
                    relations.append([rel["relation_type_id"], i, j, round(rel["weight"], 2), 0, 0])
        return relations

    def encode(self, text):
        doc = self.nlp(text)
        entities, entity_concepts, relations, entity_index = [], [], [], {}

        for chunk in doc.noun_chunks:
            head_text = chunk.root.lemma_.lower()
            concept = self._resolve_concept(head_text, text)
            modifiers = []
            for token in chunk:
                if token.dep_ in ("amod", "acomp") and token != chunk.root:
                    mod_concept = self._resolve_concept(token.lemma_.lower(), text)
                    if mod_concept: modifiers.append(mod_concept)
            if concept:
                if not modifiers: modifiers = self._get_bible_modifiers(concept, text)
                eda = self._concept_to_eda(concept, modifiers)
                idx = len(entities)
                entities.append(eda); entity_concepts.append(concept)
                entity_index[chunk.root.i] = idx
            for token in chunk:
                if token.dep_ == "compound" and token.i not in entity_index:
                    comp = self._resolve_concept(token.lemma_.lower(), text)
                    if comp:
                        cm = self._get_bible_modifiers(comp, text)
                        entities.append(self._concept_to_eda(comp, cm))
                        entity_concepts.append(comp)
                        entity_index[token.i] = len(entities) - 1

        for token in doc:
            if token.pos_ == "VERB" and token.i not in entity_index:
                concept = self._resolve_concept(token.lemma_.lower(), text)
                if concept:
                    entities.append(self._concept_to_eda(concept, confidence=0.85))
                    entity_concepts.append(concept)
                    entity_index[token.i] = len(entities) - 1

        for token in doc:
            if token.pos_ == "VERB":
                subj_idx = obj_idx = None
                for child in token.children:
                    if child.dep_ in ("nsubj", "nsubjpass") and child.i in entity_index:
                        subj_idx = entity_index[child.i]
                    elif child.dep_ in ("dobj", "attr") and child.i in entity_index:
                        obj_idx = entity_index[child.i]
                if obj_idx is None:
                    for child in token.children:
                        if child.dep_ == "prep":
                            for gc in child.children:
                                if gc.dep_ == "pobj" and gc.i in entity_index:
                                    obj_idx = entity_index[gc.i]; break
                if subj_idx is not None and obj_idx is not None:
                    rel_type = 24
                    for child in token.children:
                        if child.dep_ == "prep" and child.text.lower() in ("on", "in", "at", "near", "under"):
                            rel_type = 6; break
                    vmap = {"be": 1, "have": 3, "cause": 7, "use": 12, "make": 13, "want": 18, "need": 8, "eat": 12}
                    if token.lemma_.lower() in vmap: rel_type = vmap[token.lemma_.lower()]
                    temporal = 0
                    if token.morph.get("Tense"):
                        t = token.morph.get("Tense")[0]
                        temporal = 1 if t == "Past" else (2 if t == "Pres" else 0)
                    relations.append([rel_type, subj_idx, obj_idx, 0.85, temporal, 0])

        bible_rels = self._find_bible_relations(entity_concepts)
        existing = {(r[1], r[2]) for r in relations}
        for ra in bible_rels:
            if (ra[1], ra[2]) not in existing: relations.append(ra)

        if not entities:
            for token in doc:
                if token.pos_ in ("NOUN", "PROPN"):
                    concept = self._resolve_concept(token.lemma_.lower(), text)
                    if concept:
                        entities.append(self._concept_to_eda(concept))
                        entity_concepts.append(concept)
            if len(entities) > 1:
                for ra in self._find_bible_relations(entity_concepts): relations.append(ra)

        if not entities:
            entities.append(self._make_unknown_eda(text[:20]))

        return format_sml_block(entities, relations)


# Quick test
enc = SMLEncoder(BIBLE_DB_PATH)
test_sml = enc.encode("Can dogs bark?")
print(f"Encoder test:\n{test_sml}")
enc.close()

## 4. Define SML Harness Wrapper

This subclasses lm-eval's `HFLM` to inject SML context into every benchmark prompt.

In [ ]:
import functools
import logging
from tqdm import tqdm

from lm_eval.api.registry import register_model
from lm_eval.models.huggingface import HFLM

logger = logging.getLogger("sml_harness")


@register_model("sml_hf")
class SMLAugmentedHFLM(HFLM):
    """HFLM that injects SML context into every prompt."""

    def __init__(self, bible_path=None, spacy_model="en_core_web_sm",
                 sml_cache=4096, max_encode=2048, **kwargs):
        super().__init__(**kwargs)
        bible = bible_path or BIBLE_DB_PATH
        self._sml_encoder = SMLEncoder(bible, spacy_model=spacy_model)
        self._max_encode = int(max_encode)
        self._encode_cached = functools.lru_cache(maxsize=int(sml_cache))(
            self._sml_encoder.encode
        )
        self._warned = False
        print(f"SML encoder ready (bible={bible}, max_encode={self._max_encode})")

    def _get_snippet(self, text):
        return text[-self._max_encode:] if len(text) > self._max_encode else text

    def _encode_batch(self, requests, extract_fn):
        snippets = [self._get_snippet(extract_fn(req)) for req in requests]
        unique = list(set(snippets))
        sml_map = {}
        t0 = time.time()
        for snippet in tqdm(unique, desc=f"SML encode ({len(unique)} unique / {len(snippets)} total)"):
            try:
                sml_map[snippet] = self._encode_cached(snippet)
            except Exception:
                if not self._warned:
                    logger.warning("SML encoding failed for a prompt", exc_info=True)
                    self._warned = True
                sml_map[snippet] = ""
        elapsed = time.time() - t0
        ci = self._encode_cached.cache_info()
        print(f"  Encoded {len(unique)} unique prompts in {elapsed:.1f}s "
              f"(cache: {ci.hits} hits / {ci.misses} misses)")
        return [sml_map[s] for s in snippets]

    def loglikelihood(self, requests, disable_tqdm=False):
        sml_blocks = self._encode_batch(requests, lambda r: r.args[0])
        for req, sml in zip(requests, sml_blocks):
            if sml:
                ctx, cont = req.args
                req.arguments = (sml + "\n" + ctx, cont)
        return super().loglikelihood(requests, disable_tqdm)

    def loglikelihood_rolling(self, requests, disable_tqdm=False):
        sml_blocks = self._encode_batch(requests, lambda r: r.args[0])
        for req, sml in zip(requests, sml_blocks):
            if sml:
                (text,) = req.args
                req.arguments = (sml + "\n" + text,)
        return super().loglikelihood_rolling(requests, disable_tqdm)

    def generate_until(self, requests, disable_tqdm=False):
        sml_blocks = self._encode_batch(requests, lambda r: r.args[0])
        for req, sml in zip(requests, sml_blocks):
            if sml:
                ctx, gen_kwargs = req.args
                req.arguments = (sml + "\n" + ctx, gen_kwargs)
        return super().generate_until(requests, disable_tqdm)

    def __del__(self):
        if hasattr(self, "_sml_encoder"):
            self._sml_encoder.close()


print("SMLAugmentedHFLM registered as 'sml_hf'.")

## 5. Configuration

Set your parameters here. Use `LIMIT` for quick iteration, set to `None` for full eval.

In [ ]:
# ── Eval parameters ────────────────────────────────────────────────────
MODEL_ID     = "sweetpapa/sml-qwen2.5-3b-phase2"
TASKS        = ["arc_challenge", "hellaswag", "truthfulqa_mc2", "winogrande"]
NUM_FEWSHOT  = 5
BATCH_SIZE   = 8           # for baseline (no SML)
SML_BATCH    = "auto"      # SML adds tokens to every prompt, so auto-detect to avoid OOM
DEVICE       = "cuda:0"
DTYPE        = "float16"

# Set to an integer (e.g. 100) for quick test, None for full eval
LIMIT        = 100  # <-- change to None for full run

print(f"Model:     {MODEL_ID}")
print(f"Tasks:     {TASKS}")
print(f"Few-shot:  {NUM_FEWSHOT}")
print(f"Batch:     {BATCH_SIZE} (baseline) / {SML_BATCH} (SML)")
print(f"Limit:     {LIMIT or 'FULL'}")
print(f"Device:    {DEVICE}")

## 6. Run Baseline (No SML)

In [ ]:
import lm_eval
import json

print("=" * 70)
print("  BASELINE EVAL (no SML injection)")
print("=" * 70)

t0 = time.time()

baseline_results = lm_eval.simple_evaluate(
    model="hf",
    model_args=f"pretrained={MODEL_ID},dtype={DTYPE}",
    tasks=TASKS,
    num_fewshot=NUM_FEWSHOT,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    limit=LIMIT,
)

baseline_time = time.time() - t0
print(f"\nBaseline completed in {baseline_time:.0f}s ({baseline_time/60:.1f} min)")

# Print results
for task, data in baseline_results["results"].items():
    print(f"\n  {task}:")
    for k, v in sorted(data.items()):
        if isinstance(v, float): print(f"    {k}: {v:.4f}")

# Free GPU memory before SML run
del baseline_results["samples"]  # large, not needed
import gc; gc.collect()
torch.cuda.empty_cache()

## 7. Run SML-Augmented Eval

In [ ]:
print("=" * 70)
print("  SML-AUGMENTED EVAL (Bible context injected)")
print("=" * 70)

t0 = time.time()

sml_results = lm_eval.simple_evaluate(
    model="sml_hf",
    model_args=f"pretrained={MODEL_ID},dtype={DTYPE},bible_path={BIBLE_DB_PATH}",
    tasks=TASKS,
    num_fewshot=NUM_FEWSHOT,
    batch_size=SML_BATCH,
    device=DEVICE,
    limit=LIMIT,
)

sml_time = time.time() - t0
print(f"\nSML eval completed in {sml_time:.0f}s ({sml_time/60:.1f} min)")

for task, data in sml_results["results"].items():
    print(f"\n  {task}:")
    for k, v in sorted(data.items()):
        if isinstance(v, float): print(f"    {k}: {v:.4f}")

del sml_results["samples"]
gc.collect(); torch.cuda.empty_cache()

## 8. Comparison

In [ ]:
def extract_scores(results):
    scores = {}
    for task, data in results["results"].items():
        for k, v in data.items():
            if k.startswith("acc") and isinstance(v, float):
                scores[f"{task}/{k}"] = v
    return scores

b_scores = extract_scores(baseline_results)
s_scores = extract_scores(sml_results)

print("=" * 78)
print("  SML vs. Baseline Comparison")
print("=" * 78)
print(f"  {'Metric':<42s} {'Baseline':>10s} {'SML':>10s} {'Delta':>10s}")
print(f"  {'-' * 74}")

all_keys = sorted(set(list(b_scores.keys()) + list(s_scores.keys())))
for key in all_keys:
    b = b_scores.get(key)
    s = s_scores.get(key)
    if b is not None and s is not None:
        delta = s - b
        sign = "+" if delta >= 0 else ""
        print(f"  {key:<42s} {b:>10.4f} {s:>10.4f} {sign}{delta:>9.4f}")

print(f"\n  Baseline time: {baseline_time:.0f}s  |  SML time: {sml_time:.0f}s  |  Overhead: {sml_time - baseline_time:.0f}s")

## 9. Save Results

In [ ]:
from datetime import datetime

run_id = datetime.now().strftime("%Y-%m-%d_%H%M%S")
output = {
    "run_id": run_id,
    "model": MODEL_ID,
    "tasks": TASKS,
    "num_fewshot": NUM_FEWSHOT,
    "limit": LIMIT,
    "baseline_results": baseline_results["results"],
    "sml_results": sml_results["results"],
    "baseline_time_s": round(baseline_time, 1),
    "sml_time_s": round(sml_time, 1),
    "comparison": {k: {"baseline": b_scores.get(k), "sml": s_scores.get(k)} for k in all_keys},
}

out_path = f"/content/bench_{run_id}.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2, default=str)

print(f"Results saved to {out_path}")

# Download link in Colab
try:
    from google.colab import files
    files.download(out_path)
except ImportError:
    pass